# Deep Learning Project: From Local Receptive Fields to Global Context
### A Systematic Study of 2D Self-Attention Placement in VGG for CIFAR-10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Karakan0112/Deep-Learning-Project/blob/main/notebooks/Project_Walkthrough.ipynb)

---
## 1. Setup & Environment
Verify GPU availability and set up project imports.

In [ ]:
import os
import sys

# If running in Google Colab, automatically clone the repo
try:
    import google.colab
    if not os.path.exists('Deep-Learning-Project') and not os.path.exists('../src'):
        !git clone https://github.com/Karakan0112/Deep-Learning-Project.git
        %cd Deep-Learning-Project/notebooks
except ImportError:
    pass

import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')


## 2. CIFAR-10 Data Pipeline
Load data with random crop (padding=4), random horizontal flip, and normalization.

In [ ]:
from src.data.dataset import get_cifar10_dataloaders, CLASSES, denormalize_image

loaders = get_cifar10_dataloaders(data_dir='../data', batch_size=128, num_workers=0)
print('Train batches:', len(loaders['train']))
print('Validation batches:', len(loaders['val']))
print('Test batches:', len(loaders['test']))

# Preview sample batch
images, labels = next(iter(loaders['train']))
fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for i in range(6):
    axes[i].imshow(denormalize_image(images[i]))
    axes[i].set_title(CLASSES[labels[i]])
    axes[i].axis('off')
plt.suptitle('CIFAR-10 Sample Augmented Images', fontsize=12, fontweight='bold')
plt.show()


## 3. Model Architecture & Attention Mechanism
Inspect parameter counts and spatial dimensions across our 4 ablation models.

In [ ]:
from src.models import build_model

dummy = torch.randn(2, 3, 32, 32)
print(f'{"Model Variant":<16} | {"Parameters":<12} | {"Attention Map Shape"}')
print('-' * 55)
for m_name in ['baseline', 'attn_block1', 'attn_block2', 'attn_block3']:
    model = build_model(m_name)
    params = sum(p.numel() for p in model.parameters())
    if m_name == 'baseline':
        attn_str = 'None (Standard Conv)'
    else:
        _, attn = model(dummy, return_attention=True)
        attn_str = str(list(attn.shape[1:]))
    print(f'{m_name:<16} | {params:<12,} | {attn_str}')


## 4. Run Ablation Experiments
Run all 4 models sequentially with identical seeds, Early Stopping, and checkpointing.

In [ ]:
# Execute experiments script (on GPU, full 35 epochs takes ~10-15 mins)
!python ../experiments/run_experiments.py --epochs 35 --batch_size 128 --output_dir ../results


## 5. Quantitative Results Table & Convergence Curves

In [ ]:
import pandas as pd
from IPython.display import Image, display

csv_path = '../results/experiment_summary.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print('=== ABLATION STUDY RESULTS ===')
    display(df)

curves_img = '../results/learning_curves.png'
if os.path.exists(curves_img):
    display(Image(curves_img))


## 6. Interpretability: Comparative Attention Maps vs. Gradient Saliency Maps
Visualize local receptive fields vs. global attention context.

In [ ]:
interp_img = '../results/attention_vs_saliency.png'
if os.path.exists(interp_img):
    display(Image(interp_img))
